# Data Space I: From Observed Samples to Learnable Relationships

## 0. Learning objectives and lesson structure

We introduce the **data space** through a deliberately simple one-dimensional problem:

$$
X=\text{observed tilt angle},
\qquad
Y=\text{observed power output}.
$$

The central question is:

$$
\text{What does a finite dataset }\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n}\text{ actually tell us about the input-output relationship?}
$$

A dataset does not directly reveal the physical function. It gives finite observations from a data-generating process:

$$
(X,Y)\sim P_{\mathrm{obs}}(X,Y).
$$

By the end, students should be able to:
- distinguish a latent physical relationship from an observed dataset;
- explain why data often identify \(P(Y\mid X)\), not necessarily a deterministic function \(Y=f(X)\);
- describe how noise, missing variables, and hidden context affect observability;
- explain where a finite dataset constrains a learned function and where it does not;
- define coverage and local resolution in a one-dimensional data space;
- reason about which parts of a function need to be resolved for a given task;
- explain how confounding can distort an observed curve;
- distinguish observational prediction, deployment prediction, and causal or interventional claims;
- describe what extra data would be needed to resolve ambiguity.

The lesson is organised around three data-space questions:

1. **Observability:** what information is visible in the data?
2. **Coverage and resolution:** where does the dataset constrain behaviour?
3. **Identifiability and robustness:** does the data rule out the wrong explanations?

In [71]:
# Setup cell for the notebook.
#
# This cell should eventually:
# - import numpy, matplotlib, and IPython display utilities;
# - optionally import ipywidgets for sliders and preset buttons;
# - define a shared tilt grid theta_grid over [0, 90] degrees;
# - define a reusable latent power curve f_0(theta);
# - create plotting helpers for scatter plots, curves, coverage shading, and diagnostics;
# - set a fixed random seed so classroom examples are repeatable.
#
# Interaction notes:
# - Run this once at the start of the notebook.
# - Later cells should keep the latent curve fixed while changing the observation process.
# - Keep generated datasets small enough that repeated resampling is visually clear.

## 1. From latent physical relationships to observed datasets

Begin with a collected dataset

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n},
$$

where

$$
x_i=\text{observed tilt angle},
\qquad
y_i=\text{observed power output}.
$$

A learner fits a prediction rule

$$
h:X\rightarrow Y,
\qquad
\widehat{y}=h(x).
$$

At first glance this is ordinary one-dimensional regression. The data-space point is that the scatter plot is not the function. It is a finite, noisy, and possibly biased trace of a data-generating process.

Introduce latent variables:

$$
\Theta=\text{true tilt angle},
\qquad
C=\text{hidden context}.
$$

A useful observation model is

$$
X=\Theta+\varepsilon_X,
$$

$$
Y=f(\Theta,C)+\varepsilon_Y,
$$

and a sampling indicator

$$
S\sim\operatorname{Bernoulli}(\pi(\Theta,C,Y)).
$$

The observed dataset is

$$
\mathcal{D}=\{(X_i,Y_i):S_i=1\}.
$$

The learner observes \((X,Y)\), not \((\Theta,C,\varepsilon_X,\varepsilon_Y,S)\). A dataset constrains learning only through what it observes, where it observes, and what alternative explanations it rules out.

In [72]:
# Example cell: latent process versus observed scatter plot.
#
# This cell should eventually:
# - define latent true tilt theta and hidden context C;
# - generate observed X from theta plus optional measurement noise epsilon_X;
# - generate observed Y from f(theta, C) plus optional output noise epsilon_Y;
# - optionally sample only points with S = 1 to create biased observation;
# - plot the latent curve f(theta, C) and the observed dataset D side by side.
#
# Suggested interaction:
# - Use toggles for input noise, output noise, hidden context, and sampling bias.
# - Ask students which variables the learner sees and which variables generated the data.
# - Keep the same latent curve while changing only the observation process.

## 2. One synthetic curve, many observed datasets

Use one underlying physical curve throughout the notebook:

$$
f_0(\theta)
=
\exp\left(-\frac{(\theta-40)^2}{2\cdot 15^2}\right)
+
0.15\exp\left(-\frac{(\theta-65)^2}{2\cdot 3^2}\right),
$$

with

$$
\theta\in[0,90].
$$

This curve has a broad smooth peak and a narrow local feature. That makes it useful for discussing both coverage and resolution.

In the cleanest case,

$$
X=\Theta,
\qquad
Y=f_0(\Theta).
$$

The notebook repeatedly modifies the observation process while keeping \(f_0\) fixed. For example:

$$
Y=f_0(\Theta)+\varepsilon_Y
\quad\text{adds output noise,}
$$

$$
X=\Theta+\varepsilon_X
\quad\text{adds input noise,}
$$

and

$$
Y=A_Cf_0(\Theta)+\varepsilon_Y
\quad\text{adds hidden context.}
$$

The aim is to see that different datasets can be generated from the same latent relationship.

In [73]:
# Example cell: generate many observed datasets from one latent curve.
#
# This cell should eventually:
# - implement f_0(theta) with the broad peak and narrow feature;
# - sample theta values over [0, 90];
# - generate several datasets from the same f_0:
#   clean data, output noise, input noise, hidden context, sparse sampling, and biased sampling;
# - plot all datasets with the same axes and the same underlying f_0 reference curve.
#
# Suggested interaction:
# - Use a dropdown for observation process type.
# - Use sliders for n, input noise, output noise, and sampling bias strength.
# - Ask students which dataset best reveals the latent physical relationship and why.

## 3. Observability: what can the learner see?

The first data-space question is:

$$
\text{Does the observed input }X\text{ contain the information needed to predict }Y?
$$

Even if the physical process is deterministic in latent variables,

$$
Y=f(\Theta,C),
$$

the observed relationship between \(X\) and \(Y\) may not be deterministic. With

$$
X=\Theta+\varepsilon_X,
\qquad
Y=f(\Theta,C)+\varepsilon_Y,
$$

the same observed \(x\) can correspond to many possible \(y\) values.

The relevant data-space object is the conditional distribution

$$
P(Y\mid X=x).
$$

A learned regression function is usually a summary of this conditional distribution, not the distribution itself. For squared-error prediction, the best possible predictor using only \(X\) is

$$
h^{\star}(x)=\mathbb{E}[Y\mid X=x].
$$

The irreducible uncertainty is

$$
\operatorname{Var}(Y\mid X=x).
$$

Observability determines what distinctions are possible to learn. If relevant distinctions are absent from \(X\), the model cannot recover them from \((X,Y)\) alone.

In [74]:
# Example cell: conditional distribution and irreducible uncertainty.
#
# This cell should eventually:
# - generate repeated observations at similar or identical observed tilt values X;
# - show how output noise, input noise, and hidden context widen P(Y | X = x);
# - estimate or visualise local conditional mean E[Y | X = x];
# - estimate or visualise local conditional variance Var(Y | X = x);
# - demonstrate that h*(x) under squared error is a conditional mean, not a physical curve.
#
# Suggested interaction:
# - Use a selected x-location and highlight nearby samples.
# - Toggle hidden context labels on/off to show what information is missing from X.
# - Ask what extra measurement would reduce Var(Y | X = x).

## 4. \(P(Y\mid X)\): the dataset does not necessarily define one curve

For a clean deterministic function,

$$
Y=f(X),
$$

the conditional distribution \(P(Y\mid X=x)\) is concentrated at one value:

$$
P(Y\mid X=x)=\delta_{f(x)}.
$$

With noise or hidden context, the same observed \(x\) can correspond to multiple possible \(y\) values. For example,

$$
Y=A_Cf_0(\Theta)+\varepsilon_Y,
$$

where

$$
C\in\{0,1\},
\qquad
A_0\neq A_1.
$$

If \(C\) is hidden, the learner sees

$$
P(Y\mid X=x),
$$

not

$$
P(Y\mid X=x,C=c).
$$

Useful example observation models are:

$$
\text{clean:}\qquad X=\Theta,\quad Y=f_0(\Theta),
$$

$$
\text{output noise:}\qquad X=\Theta,\quad Y=f_0(\Theta)+\varepsilon_Y,
$$

$$
\text{input noise:}\qquad X=\Theta+\varepsilon_X,\quad Y=f_0(\Theta),
$$

and

$$
\text{hidden context:}\qquad X=\Theta,\quad Y=A_Cf_0(\Theta)+\varepsilon_Y.
$$

The same scatter-plot form \(X\) versus \(Y\) can therefore represent very different kinds of uncertainty.

In [75]:
# Example cell: four datasets with the same scatter-plot axes but different uncertainty.
#
# This cell should eventually:
# - generate clean, output-noise, input-noise, and hidden-context datasets;
# - plot each as X versus Y with the same axis limits;
# - optionally show conditional bands or local summaries for P(Y | X = x);
# - colour by hidden context only when the instructor toggles labels on.
#
# Suggested interaction:
# - Use a four-panel plot or a dropdown to switch scenarios.
# - Ask students whether spread around the curve is label noise, input noise,
#   hidden context, or something a larger model could fix.
# - Ask whether the model estimates a physical curve or an average over hidden contexts.

## 5. Coverage: where does the dataset constrain the function?

The second data-space question is:

$$
\text{Where in the input space does the dataset provide evidence?}
$$

If observed data cover only

$$
x\in[20,60],
$$

then behaviour outside this interval is extrapolation. If data are dense near \(x=40\) but sparse near \(x=65\), the broad peak may be constrained while the narrow feature may not be.

Define the local sample count within radius \(r\):

$$
n_r(x)=\sum_{i=1}^{n}\mathbf{1}\!\left\{|x_i-x|\leq r\right\}.
$$

Coverage is about support and density under the training distribution

$$
p_{\mathrm{train}}(x).
$$

A basic condition for learning behaviour at deployment is

$$
p_{\mathrm{deploy}}(x)>0
\implies
p_{\mathrm{train}}(x)>0,
$$

at least in regions where errors matter.

Coverage alone is not enough, but without coverage the data cannot directly constrain the function.

In [76]:
# Example cell: support, density, and extrapolation regions.
#
# This cell should eventually:
# - generate datasets with the same sample count but different sampling patterns:
#   uniform over [0, 90], clustered near the broad peak, missing the narrow feature,
#   sparse near x = 65, and restricted to [20, 60];
# - compute local sample count n_r(x) over a grid;
# - plot the observed points, latent curve, shaded unsupported regions, and n_r(x);
# - fit the same simple smoother to each dataset to show where model bias fills gaps.
#
# Suggested interaction:
# - Use a dropdown for sampling pattern and a slider for radius r.
# - Ask which parts of the curve are constrained by data and which are extrapolation.
# - Compare random-test adequacy with adequacy for a specific deployment region.

## 6. Resolution and task-weighted adequacy

Coverage says whether observations exist in a region. Resolution asks whether those observations are dense and clean enough to distinguish the relevant behaviour.

Let

$$
\mu(x)=\mathbb{E}[Y\mid X=x].
$$

Define local behavioural variation over radius \(r\):

$$
\Delta_r(x)
=
\sup_{u,v\in[x-r,x+r]}|\mu(u)-\mu(v)|.
$$

This measures how much the target behaviour changes locally. A useful diagnostic is

$$
\rho_r(x)
=
\frac{\Delta_r(x)}{\widehat{\sigma}(x)/\sqrt{n_r(x)}},
$$

where

$$
\widehat{\sigma}(x)^2\approx\widehat{\operatorname{Var}}(Y\mid X=x).
$$

Interpretation:

- large \(\rho_r(x)\): local behaviour is likely distinguishable;
- small \(\rho_r(x)\): local behaviour may be hidden by noise or sparse sampling.

This is a diagnostic, not a universal theorem.

A dataset also need not resolve the function equally everywhere. Let \(w(x)\geq 0\) be a task-importance function. A task-weighted squared risk is

$$
R_w(h)=\mathbb{E}\left[w(X)(h(X)-Y)^2\right].
$$

The practical question becomes:

$$
\text{Does the dataset resolve }\mu(x)\text{ where }w(x)\text{ is large?}
$$

In [77]:
# Example cell: resolution near the narrow feature and task-weighted adequacy.
#
# This cell should eventually:
# - focus on the narrow feature near x = 65;
# - compare sparse/noisy, dense/noisy, sparse/low-noise, and dense/low-noise datasets;
# - compute or visualise n_r(x), local variation Delta_r(x), local noise sigma_hat(x),
#   and the diagnostic rho_r(x);
# - evaluate the same fitted model under several task weights w(x): uniform, broad-peak,
#   narrow-feature, and edge-case importance.
#
# Suggested interaction:
# - Use sliders for local radius r, noise level, and sample density near x = 65.
# - Use a dropdown for the task-weighting function w(x).
# - Ask why a dataset can be adequate for one task and inadequate for another.

## 7. Identifiability: does the data rule out the wrong explanations?

The third data-space question is:

$$
\text{Does the dataset distinguish the intended relationship from alternative explanations?}
$$

A finite dataset rarely identifies a unique function. For a hypothesis space \(\mathcal{H}\), define the set of good-fitting hypotheses

$$
\mathcal{V}_{\varepsilon}(\mathcal{D})
=
\left\{
 h\in\mathcal{H}:
 \widehat{R}(h)
 \leq
 \inf_{h'\in\mathcal{H}}\widehat{R}(h')+\varepsilon
\right\},
$$

where

$$
\widehat{R}(h)=\frac{1}{n}\sum_{i=1}^{n}(h(x_i)-y_i)^2.
$$

The dataset has identified behaviour near \(x\) if all good-fitting hypotheses behave similarly there. Define disagreement

$$
\operatorname{Dis}(x)
=
\sup_{h,h'\in\mathcal{V}_{\varepsilon}(\mathcal{D})}|h(x)-h'(x)|.
$$

If \(\operatorname{Dis}(x)\) is large in a region we care about, the data has not identified the function there.

Similar training error does not mean the data has identified the intended behaviour.

In [78]:
# Example cell: model disagreement among similarly plausible explanations.
#
# This cell should eventually:
# - fit several plausible models to the same one-dimensional data:
#   linear model, polynomial model, spline, kernel smoother, and small neural network
#   if dependencies make that practical;
# - keep models with similar training or validation error;
# - plot their predictions over [0, 90];
# - compute a simple disagreement band across retained models;
# - relate high-disagreement regions to missing data, noise, or weak validation coverage.
#
# Suggested interaction:
# - Use checkboxes to show/hide model classes.
# - Use an epsilon slider to decide which models count as similarly good-fitting.
# - Ask where the models agree, where they disagree, and what extra data would resolve it.

## 8. Hidden context, confounding, and causal claims

Introduce hidden context

$$
C\in\{0,1\}.
$$

A hidden variable is not automatically a confounder. If

$$
C\rightarrow Y
$$

but not \(X\), then \(C\) creates hidden heterogeneity. If

$$
C\rightarrow X
\quad\text{and}\quad
C\rightarrow Y,
$$

then \(C\) is a confounder.

In the tilt-power setup, confounding means that the distribution of hidden context changes with tilt angle:

$$
P(C=1\mid X=x)
$$

may be different at low and high tilt angles.

The observed curve is

$$
\mu_{\mathrm{obs}}(x)=\mathbb{E}[Y\mid X=x].
$$

Expanding over hidden context,

$$
\mathbb{E}[Y\mid X=x]
=
\sum_c \mathbb{E}[Y\mid X=x,C=c]P(C=c\mid X=x).
$$

The interventional or physical tilt-response curve is closer to

$$
\mu_{\mathrm{causal}}(x)=\mathbb{E}[Y\mid do(\Theta=x)].
$$

Under suitable assumptions,

$$
\mathbb{E}[Y\mid do(\Theta=x)]
=
\sum_c \mathbb{E}[Y\mid \Theta=x,C=c]P(C=c).
$$

Observational conditioning uses \(P(C=c\mid X=x)\); intervention uses \(P(C=c)\). The observed curve may therefore mix the tilt effect with a changing hidden-context distribution.

In [79]:
# Example cell: hidden context versus confounding.
#
# This cell should eventually:
# - generate two contexts C = 0 and C = 1 with different amplitudes A_C;
# - use Y = A_C * f_0(theta) + epsilon;
# - first make C independent of theta to show hidden heterogeneity;
# - then make P(C = 1 | theta) increase with theta to show confounding;
# - plot observed scatter without context labels, scatter coloured by context,
#   context-specific curves, and the observed conditional mean.
#
# Suggested interaction:
# - Toggle context labels on/off.
# - Toggle whether context is independent of theta or depends on theta.
# - Ask whether the observed curve is a physical tilt-response curve or an observational mix.
# - Ask what additional measurement or design would identify the causal curve.

## 9. Robustness and link to hypothesis and optimisation spaces

A model may perform well under a random split because train and test share the same hidden-context structure. Let

$$
P_{\mathrm{train}}(X,Y)
$$

and

$$
P_{\mathrm{deploy}}(X,Y)
$$

be the training and deployment distributions. Random validation estimates performance under \(P_{\mathrm{train}}\), but deployment may follow

$$
P_{\mathrm{deploy}}(X,Y)\neq P_{\mathrm{train}}(X,Y).
$$

In the hidden-context example, train and deployment may differ in

$$
P(C\mid X),
$$

which changes

$$
P(Y\mid X)
$$

even if the context-specific functions stay the same.

Useful test distributions include:

$$
P_{\mathrm{test}}(C\mid X)=P_{\mathrm{train}}(C\mid X)
\quad\text{for a random split,}
$$

$$
P_{\mathrm{test}}(C=0\mid X)=P_{\mathrm{test}}(C=1\mid X)=0.5
\quad\text{for balanced context,}
$$

and a shifted or reversed \(P_{\mathrm{test}}(C\mid X)\) for deployment stress tests.

Expected pattern:

$$
\text{random split error}
<
\text{balanced or shifted test error}.
$$

The point is not simply that the model is bad. The point is that the dataset did not identify a relationship stable across changes in hidden context.

Summary:

$$
\mathcal{D}\text{ determines what is evidenced,}
$$

$$
\mathcal{H}\text{ determines what functions are possible,}
$$

and

$$
\mathcal{O}\text{ determines which compatible function is selected.}
$$

The next notebook asks how the hypothesis space shapes interpolation and extrapolation when data are finite and imperfect.

In [80]:
# Optional wrap-up and robustness example cell.
#
# This cell should eventually:
# - fit one model on the confounded training distribution;
# - evaluate it on three test sets: random split, balanced context, and shifted/reversed context;
# - report errors in a compact table;
# - plot predictions against each test distribution;
# - connect observed failures back to D, H, and O.
#
# Suggested interaction:
# - Use controls for the deployment context shift strength.
# - Ask whether a more expressive model, stronger regularisation, observing C,
#   randomising X, or collecting balanced data would address the failure.
# - Use this as the handoff to the hypothesis-space notebook.